In [1]:
!pip install -q unsloth
!pip install bert-score          #Устанавливаю библиотеки для проведения дообучения и получения Bert метрик

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9

In [2]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 2048 ,
    dtype = None,                                                       #В Параметрах всего буду использовать в основном базовые стандартные значения , скачиваю квантизированную версию этой ЛЛМ
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [3]:
model = FastLanguageModel.get_peft_model(
    model,                                           #Также в основном выставляю либо усредненные либо стандартные значения для параметров
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", #Во время дообучения будут затронуты все Attention и Feed-Forward слои для лучших результатов
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16, #Стандартный фактор шкалы 1
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 2101,
    use_rslora = False,   # из за лоры альфы выставляю на False
    loftq_config = None,
)

Unsloth 2026.4.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [58]:
from datasets import load_dataset
dataset = load_dataset("theblackcat102/joke_explaination", split="train")   # Загружаю датасет по объяснению шуток

In [59]:
dataset[53]

{'url': 'https://explainthejoke.com/2021/10/31/all-wound-up/',
 'joke': 'Q: Why don’t mummies take vacations from their work? A: Because they are afraid to unwind!',
 'explaination': 'Explanation: Most people enjoy taking a few days off from school or work to rest and relax, to unwind. In this sense, to unwind means to relax after work, after school, or after a stressful time.\nTo unwind also means to undo or open up something that has been wound up. Think of a ball of string– when you pull the loose end of the string you unwind it. You can also unwind an electrical extension cord, cassette tape, Christmas lights, ….\nMummies are wrapped up in cloth; if you start to take the cloth off the mummy in one long strip you are unwinding the cloth.\nThis joke is funny because it uses unwind in two different ways: to relax and to uncoil something.\nWith the following video you can learn to wind up and throw a traditional top.'}

In [60]:
import re                                    # Очистка и препроцессинг датасета .
def remove_q_a(example):
    example["joke"] = re.sub(r'\s*(Q:|A:)\s*', '', example["joke"])
    return example

dataset = dataset.map(remove_q_a)
seen = set()
def filter_duplicates(example):
    if example["joke"] in seen:
        return False
    seen.add(example["joke"])
    return True

dataset = dataset.filter(filter_duplicates)
dataset = dataset.remove_columns("url")
dataset = dataset.rename_column("joke", "prompt")
dataset = dataset.rename_column("explaination", "completion")

In [62]:

dataset[100]

{'prompt': 'What music frightens balloons?Pop music!',
 'completion': 'Explanation: When balloons explode, we say they pop. If you stick a pin in a balloon, it pops. If balloons had feelings, I am pretty sure that they would be scared of things that make the pop.\n‘Pop’ is also a type of music, originally from the term ‘popular music.’ Pop music now refers to music that is commercial (sells a lot), usually upbeat, and uses a verse/chorus format. Yea, that’s probably too much information. (By the way, pop can also mean father, and soda.)\nThis joke is funny because it plays with the word pop–both a type of music and an action that end the life of a balloon. There is a lot of famous pop music. Here is one of the pop music examples from the list (#26) in the previous links:'}

In [ ]:
prompt = "Why don’t mummies take vacations from their work? Because they are afraid to unwind!' Explain joke"
inputs = tokenizer(prompt, return_tensors="pt").to(device='cuda')
outputs = model.generate(
    **inputs,
    max_new_tokens=2048,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [ ]:
response = tokenizer.batch_decode(outputs)
response

["<|begin_of_text|>Why don’t mummies take vacations from their work? Because they are afraid to unwind!' Explain joke: To unwind is to relax after a stressful experience or after working. When you unwind you might take a walk, read a book, listen to music, take a bath, or fall asleep. When you unwind you are relaxing, de-stressing, calming your mind and body.\nMummies are dead bodies wrapped in cloth; they are buried. It is not common for mummies to work, unless they are archaeologists studying mummies. If they work, they probably need to unwind after they finish working.\nThis joke is funny because it plays with the word unwind. Mummies are dead, so they do not need to unwind.\nHere is a mummy unwinding after a long time:<|eot_id|>"]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    max_seq_length = 2048,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Из за маленьких батчей
        warmup_steps = 5, # Регулирует learning rate
        max_steps = 180,  # Выставил 60 эпох но показалось мало
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
history = trainer.train() #Что бы сохранить данные об обучении

In [ ]:
model.save_pretrained("jokesplain-v1-3B")
tokenizer.save_pretrained("jokespain-v1-3B")

('jokespain-v1-3B/tokenizer_config.json',
 'jokespain-v1-3B/chat_template.jinja',
 'jokespain-v1-3B/tokenizer.json')

In [ ]:
import json

with open("training_history.json", "w") as f:
    json.dump(history, f)


In [6]:
import os

os.makedirs("jokesplain-v1-3B", exist_ok=True)


In [ ]:
model_fined, tokenizer_fined = FastLanguageModel.from_pretrained(    #Загружаю дообученную модель для тестирования 
    model_name = "./jokesplain-v1-3B",  
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.4.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [ ]:
model , model_fined

In [20]:
prompt = "How do you call a black man who drives plane? Just a pilot . Explain joke"
inputs = tokenizer_fined(prompt, return_tensors="pt").to(device='cuda')
outputs = model_fined.generate(
    **inputs,
    max_new_tokens=2048,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [21]:
response = tokenizer_fined.batch_decode(outputs)
response

['<|begin_of_text|>How do you call a black man who drives plane? Just a pilot. Explain joke:\nPilot is the person who flies the airplane; the pilot can be any race, any nationality, any gender. The word pilot has nothing to do with flying; pilot means to move forward, to advance.\nThis joke is funny because pilot sounds the same as racist. If you are racist you are prejudiced against someone because of race. This joke is funny because it plays with the word pilot/ racist.\nHere is a video about the history of aviation and the first female pilot:\nVIDEO\n<|eot_id|>']

In [ ]:
from datasets import load_dataset

dataset = load_dataset("theblackcat102/joke_explaination")
dataset = dataset.filter(filter_duplicates)
dataset = dataset.remove_columns("url")
dataset = dataset.rename_column("joke", "prompt")
dataset = dataset.rename_column("explaination", "completion")

split_dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

train_dataset ,test_dataset


377


(Dataset({
     features: ['prompt', 'completion'],
     num_rows: 301
 }),
 Dataset({
     features: ['prompt', 'completion'],
     num_rows: 76
 }))

In [ ]:
from bert_score import score            
  
def generate_outputs(model, tokenizer, dataset, num_samples=15):
    outputs, references = [], []
    for example in dataset.select(range(num_samples)):  
        prompt = example["prompt"]
        reference = example["completion"]

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=128)
        prediction = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

        outputs.append(prediction)
        references.append(reference)
    return outputs, references


preds_fined, refs = generate_outputs(model_fined, tokenizer_fined, test_dataset)
preds_base, _ = generate_outputs(model, tokenizer, test_dataset)

P_fined, R_fined, F1_fined = score(preds_fined, refs, lang="en", verbose=True)
P_base, R_base, F1_base = score(preds_base, refs, lang="en", verbose=True)

print("Fine-tuned model BERTScore (F1):", F1_fined.mean().item())
print("Base model BERTScore (F1):", F1_base.mean().item())

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.90 seconds, 16.59 sentences/sec
Fine-tuned model BERTScore (F1): 0.8664075136184692
Base model BERTScore (F1): 0.8306448459625244


In [45]:
results = {
    "Fine_tuned": {
        "Precision": P_fined.mean().item(),
        "Recall": R_fined.mean().item(),
        "F1": F1_fined.mean().item()
    },
    "Base": {
        "Precision": P_base.mean().item(),
        "Recall": R_base.mean().item(),
        "F1": F1_base.mean().item()
    }
}

with open("bertscore_results.json", "w") as f:
    json.dump(results, f, indent=4)

In [35]:
import json

with open("./jokesplain-v1-3B/training_history.json", "r") as f:
    metrics = json.load(f)
metrics

[180,
 1.2420098894172245,
 {'train_runtime': 339.3029,
  'train_samples_per_second': 4.244,
  'train_steps_per_second': 0.53,
  'total_flos': 4311502382592000.0,
  'train_loss': 1.2420098894172245,
  'epoch': 3.761904761904762}]

In [ ]:
with open("bertscore_results.json", "r") as f:
    results = json.load(f)
print(results)

In [ ]:
import mlflow

# Start an MLflow experiment
mlflow.set_experiment("jokesplain_eval")

with mlflow.start_run(run_name="bert_score_comparison"):
    # Log evaluation parameters
    mlflow.log_params({
        "num_samples": 15,
        "dataset": "theblackcat102/joke_explaination",
        "metric": "BERTScore",
    })

    # Generate predictions
    preds_fined, refs = generate_outputs(model_fined, tokenizer_fined, test_dataset, num_samples=15)
    preds_base, _ = generate_outputs(model, tokenizer, test_dataset, num_samples=15)

    # Compute BERTScore
    P_fined, R_fined, F1_fined = score(preds_fined, refs, lang="en", verbose=True)
    P_base, R_base, F1_base = score(preds_base, refs, lang="en", verbose=True)

    # Log metrics (mean values)
    mlflow.log_metric("Fine_tuned_P", P_fined.mean().item())
    mlflow.log_metric("Fine_tuned_R", R_fined.mean().item())
    mlflow.log_metric("Fine_tuned_F1", F1_fined.mean().item())

    mlflow.log_metric("Base_P", P_base.mean().item())
    mlflow.log_metric("Base_R", R_base.mean().item())
    mlflow.log_metric("Base_F1", F1_base.mean().item())

print("Logged BERTScore results to MLflow.")


In [ ]:
!pip install mlflow

In [38]:
metrics

[180,
 1.2420098894172245,
 {'train_runtime': 339.3029,
  'train_samples_per_second': 4.244,
  'train_steps_per_second': 0.53,
  'total_flos': 4311502382592000.0,
  'train_loss': 1.2420098894172245,
  'epoch': 3.761904761904762}]

In [40]:
!mlflow ui --port 5000


Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
INFO:     Started parent process [19009]
2026/04/05 10:12:01 INFO mlflow.server.jobs.utils: Starting huey consumer for job function run_online_session_scorer
2026/04/05 10:12:01 INFO mlflow.server.jobs.utils: Starting huey consumer for job function invoke_scorer
2026/04/05 10:12:01 INFO mlflow.server.jobs.utils: Starting huey consumer for job function optimize_prompts
2026/04/05 10:12:01 INFO mlflow.server.jobs.utils: Starting huey consumer for job function run_online_trace_scorer
2026/04/05 10:12:01 INFO mlflow.server.jobs.utils: Starting dedicated Huey consumer for periodic tasks
INFO:     Started server pro